# Predicción del caudal medio diario en cuencas de Colombia con una LSTM regional

**Proyecto final — Análisis con Deep Learning, Universidad de los Andes**
Santiago Palacios Loaiza · Diego Leonardo Barón Espitia · Andrés Felipe Mosquera Hernández · Juan Sebastián Rodríguez Avella

**Parte 1: preprocesamiento.** Este notebook lee las series y los atributos de CAMELS-COL, los limpia, los consolida en una tabla y la convierte al formato de NeuralHydrology. Todo se ejecuta aquí, paso a paso, y el resultado es una base nueva en `data/processed/`, lista para entrenar el modelo. La sección 2 agrega 15 forzantes de ERA5-Land y una segunda base en `data/processed_era5land/`. El modelado (parte 2) está en `proyecto-final-modelado.ipynb`, que parte de esa base.

> **Regla del proyecto:** el modelo **no** recibe caudal observado como entrada (ni `Q(t-1)` ni nada derivado del caudal).
> Es simulación a partir del clima, y eso es lo que permite usarlo en cuencas sin estación de aforo.

## 0. Entorno

In [1]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Lectura y definicon de rutas de fuentes y base de salida.
ROOT = Path.cwd()
DIR_DATOS = ROOT / "data" / "datos-originales"                  # CSV de atributos originales de CAMELS-COL
DIR_SERIES = DIR_DATOS / "3_Hydrometeorological_data_filtrado"  # series diarias pre-filtradas (1.2)
DIR_SALIDA = ROOT / "data" / "processed"                        # base nueva, lista para el modelo
assert DIR_SERIES.exists(), "Ejecuta el notebook desde la carpeta raíz del proyecto"

# Calendario diario común y partición temporal (justificación en 1.9)
CALENDARIO = pd.date_range("1981-01-01", "2022-12-31", freq="D")
PERIODOS = {"train": ("1981-01-01", "2010-12-31"),
            "validation": ("2011-01-01", "2015-12-31"),
            "test": ("2016-01-01", "2022-12-31")}
EN_TRAIN = (CALENDARIO >= PERIODOS["train"][0]) & (CALENDARIO <= PERIODOS["train"][1])

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

## 1. Preprocesamiento

Se leen las fuentes (1.3), se aplica cada ajuste a las 248 cuencas y al final se escribe la base nueva en `data/processed/` (1.11 y 1.12), lista para entrenar el modelo. Cada subsección describe un problema de los datos, la decisión que se tomó y el código que la aplica. 

### 1.1 Variables del modelo

El modelo simula el caudal diario (`streamflow`, en mm/día) a partir de **4 forzantes dinámicas** (variables indispensables para predecir caudal), que lee para cada día de una ventana de 365 días, y de **48 atributos estáticos**, que describen cada cuenca y no cambian en el tiempo. Nunca recibe caudal observado como entrada.

Del dataset se excluyen:
- **Firmas hidrológicas** (`q_mean`, `baseflow_index`, …): se calculan con el caudal observado. Serían fuga de información y no existen en cuencas no aforadas; se guardan solo para analizar resultados.
- **Metadatos de la estación** (fechas, departamento, coordenadas): describen la estación, no la cuenca. Con las coordenadas, el modelo podría reconocer cada cuenca por su posición en vez de aprender de sus propiedades.
- **Columnas redundantes:** `gauge_elev` (repite `minimum_ele`), `perimeter` (se deduce del área y del índice de Gravelius), 3 de las 4 fórmulas de tiempo de concentración (ordenan igual a las cuencas; se conserva `tc_Johnstone`), `volcaniclastic_rock_perc` (copia de `volcanic_rock_perc`), `Urban_area` (misma clase que `urban_perc`, medida con otro mapa) y `No_classified` (equivale a los cuerpos de agua).

In [2]:
# Forzantes dinámicas: columna en los archivos -> nombre en el proyecto
FORZANTES = {"Precipitacion": "pr", "ETP_": "poten_evapo",
             "Temperatura_minima": "t_min", "Temperatura_maxima": "t_max"}
OBJETIVO = "streamflow"   # caudal en mm/día (1.6)

# Atributos estáticos candidatos. log_area se calcula en 1.7 y los climáticos en 1.8;
# el resto se toma de los CSV de CAMELS-COL. En 1.7 se descartan los casi vacíos.
ATRIBUTOS_CANDIDATOS = {
    "fisiograficos": ["log_area", "gravelius_index", "factor_form", "maximum_ele", "mean_ele", "minimum_ele",
                      "streng_chanel", "equi_slope", "cn_catchment",
                      "tc_Johnstone"],   # reemplaza a tc_representative (1.7)
    "climaticos": ["p_mean", "pet_mean", "aridity", "high_prec_freq", "high_prec_dur",
                   "low_prec_freq", "low_prec_dur"],
    "cobertura": ["forest_perc", "nat_non_forest_form_perc", "agricul_livestock_perc", "non_vegeted_perc",
                  "water_bodies_perc_cob", "non_identi_land_perc"],
    "geologia": ["unconso_depo_perc", "hypabyssal_rock_perc", "metamor_rock_perc", "plutonic_rock_perc",
                 "sedimen_rock_perc", "volcanic_rock_perc"],   # volcaniclastic_rock_perc es copia de volcanic
    "suelos": ["inceptisols_perc", "entisols_perc", "andisols_perc", "alfisols_perc", "vertisols_perc",
               "ultisols_perc", "oxisols_perc", "mollisols_perc", "aridisols_perc", "histosols_perc",
               "urban_perc", "eroded_misce_perc", "rocky_misce_perc", "water_bodies_perc_suelo"],
    "capacidad_uso": [f"class_{i}" for i in range(1, 9)],
}

# Se guardan en la tabla de atributos, pero no entran al modelo
AUXILIARES = ["region", "area", "gauge_lat", "gauge_lon", "gauge_department", "gauge_star", "gauge_end", "gauge_missing"]
FIRMAS_HIDROLOGICAS = ["q_mean", "runoff_ratio", "stream_elas", "slope_fdc", "baseflow_index", "Q5", "Q95",
                       "high_q_freq", "high_q_dur", "low_q_freq", "low_q_dur"]

print(f"{len(FORZANTES)} forzantes y {sum(map(len, ATRIBUTOS_CANDIDATOS.values()))} atributos candidatos:",
      {g: len(cols) for g, cols in ATRIBUTOS_CANDIDATOS.items()})

4 forzantes y 51 atributos candidatos: {'fisiograficos': 10, 'climaticos': 7, 'cobertura': 6, 'geologia': 6, 'suelos': 14, 'capacidad_uso': 8}


### 1.2 Pre-filtrado de las variables dinámicas

Las series diarias (caudal y forzantes) **no son las originales del repositorio Zenodo**. Antes, se aplicó un control de calidad propio que dejó 248 de las 346 cuencas y corrigió la evapotranspiración. El resultado está en `data/datos-originales/3_Hydrometeorological_data_filtrado/`, junto con `cleaning_report.csv`, que registra qué se hizo con cada cuenca. Mientras el código del filtrado no esté en este notebook, el procesamiento parte de esas series ya filtradas.

> ✏️ **Por completar: explicación del pre-filtrado** (Santi)
>
> Describir a grandes rasgos los pasos seguidos. Puntos sugeridos:
> 1. Qué detecto y con qué criterio se aplicaron los cambios (tabla de abajo).
> 2. Cómo se detectaron los picos (`spike`) y los valores congelados (`frozen`) del caudal.
> 3. Cómo y por qué se recalculó la evapotranspiración potencial.
> 4. El código del filtrado, en la celda de abajo, para que el notebook lo ejecute y el resultado se pueda reproducir desde los datos de Zenodo.

| Bandera | Cuencas descartadas con esta bandera | Qué detecta y con qué criterio |
|---|---|---|
| `f_q_gt_p` | 55 | *por completar* |
| `f_regime_mismatch` | 20 | *por completar* |
| `f_flatline` | 19 | *por completar* |
| `f_nested_inconsistent` | 13 | *por completar* |
| `f_delineation` | 6 | *por completar* |
| `f_twin_mismatch` | 5 | *por completar* |

In [3]:
# ✏️ Por completar: código del pre-filtrado. Debe leer las series originales de Zenodo
# y escribir las series filtradas y cleaning_report.csv en DIR_SERIES.

In [4]:
print(DIR_SERIES / "cleaning_report.csv")

C:\Users\Diego\Documents\ADL\proyecto-final\data\datos-originales\3_Hydrometeorological_data_filtrado\cleaning_report.csv


In [5]:
# lee el reporte de limpieza y muestra un resumen
reporte = pd.read_csv(DIR_SERIES / "cleaning_report.csv", dtype={"gauge_id": str})
conservadas = reporte[reporte["kept"]]
print(f"Cuencas en CAMELS-COL: {len(reporte)} | conservadas: {len(conservadas)} | descartadas: {len(reporte) - len(conservadas)}")
print(f"Días de caudal anulados en las conservadas: {int(conservadas['spike_days'].sum())} picos, {int(conservadas['frozen_days'].sum())} congelados")
print(f"Evapotranspiración media: {conservadas['pet_mean_before'].mean():.1f} → {conservadas['pet_mean_after'].mean():.1f} mm/día")
print()
pd.crosstab(reporte["region"], reporte["kept"].map({True: "conservadas", False: "descartadas"}),
            margins=True, margins_name="Total")

Cuencas en CAMELS-COL: 346 | conservadas: 248 | descartadas: 98
Días de caudal anulados en las conservadas: 399 picos, 2767 congelados
Evapotranspiración media: 8.9 → 3.3 mm/día



kept,conservadas,descartadas,Total
region,,,
Amazon,11,11,22
Caribbean,44,9,53
Magdalena-Cauca,136,46,182
Orinoco,37,20,57
Pacific,20,12,32
Total,248,98,346


### 1.3 Lectura de las fuentes

Se leen dos fuentes: las series diarias pre-filtradas (1.2), un archivo por cuenca, y los CSV de atributos originales de CAMELS-COL (archivos 02 y 05 a 11). El archivo 08, de índices climáticos, no se usa porque los índices se recalculan (1.8).

- **Series.** (variables dinámicas) Cada serie se lleva a un calendario diario continuo de 1981 a 2022. Los días que no están en el archivo quedan como filas vacías con estado `sin_registro`; así cada ventana de 365 filas corresponde a 365 días consecutivos.
- **Atributos.** Los CSV tienen inconsistencias de formato que, sin tratamiento, producen errores silenciosos: codificación Latin-1 (02), espacios al final de los nombres de columna (10), 16 384 filas de relleno e identificadores con decimales (11) y la columna `water_bodies_perc` repetida en cobertura y suelos (06 y 07). `leer_csv` las corrige, y `cargar_atributos` valida que no haya identificadores ni columnas repetidas y que todas las cuencas tengan atributos.

In [6]:
def leer_serie(ruta):
    """Lee la serie de una cuenca y la reindexa al calendario diario continuo."""
    df = pd.read_csv(ruta, sep="\t")
    df.index = pd.to_datetime(df.pop("Fecha"), format="%d/%m/%Y")
    if df.index.duplicated().any():
        raise ValueError(f"Fechas repetidas en {ruta.name}")
    df = df.rename(columns={**FORZANTES, "Caudal": "streamflow_m3s", "Caudal_estado": "streamflow_estado"})
    df = df.reindex(CALENDARIO).rename_axis("date")
    df["streamflow_estado"] = df["streamflow_estado"].fillna("sin_registro")
    return df


archivos = sorted(DIR_SERIES.glob("Hydromet_data_*.txt"))
series = {a.stem.removeprefix("Hydromet_data_"): leer_serie(a) for a in archivos}
ids = list(series)
print(f"Series: {len(series)} cuencas x {len(CALENDARIO):,} días")

Series: 248 cuencas x 15,340 días


In [7]:
# Archivo de atributos -> columnas a renombrar (water_bodies_perc está en dos archivos)
ARCHIVOS_ATRIBUTOS = {
    "02_CAMELS_COL_Catchment_information.csv": {},
    "05_CAMELS_COL_Geologic_characteristics.csv": {},
    "06_CAMELS_COL_Land_cover_characteristics.csv": {"water_bodies_perc": "water_bodies_perc_cob"},
    "07_CAMELS_COL_Soil_characteristics.csv": {"water_bodies_perc": "water_bodies_perc_suelo"},
    "09_CAMELS_COL_Hydrological_signatures.csv": {},
    "10_CAMELS_COL_Physiograpic_characteristics.csv": {},
    "11_CAMELS_COL_Land_use_capability.csv": {},
}


def leer_csv(ruta):
    """Lee un CSV de atributos corrigiendo las inconsistencias de formato del dataset."""
    crudo = ruta.read_bytes()
    try:
        texto = crudo.decode("utf-8")
    except UnicodeDecodeError:
        texto = crudo.decode("latin-1")
    encabezado = texto.splitlines()[0]
    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    df = pd.read_csv(io.StringIO(texto), sep=sep, dtype={"gauge_id": str})
    df.columns = df.columns.str.strip()
    df = df.dropna(subset=["gauge_id"])                                                  # filas de relleno
    df["gauge_id"] = df["gauge_id"].str.strip().str.replace(r"\.0+$", "", regex=True)   # "11017010.00"
    if df["gauge_id"].duplicated().any():
        raise ValueError(f"gauge_id repetidos en {ruta.name}")
    return df.set_index("gauge_id")


def cargar_atributos(ids):
    """Une los archivos de atributos en una tabla con una fila por cuenca."""
    attrs = pd.concat([leer_csv(DIR_DATOS / nombre).rename(columns=renombres)
                       for nombre, renombres in ARCHIVOS_ATRIBUTOS.items()], axis=1)
    repetidas = attrs.columns[attrs.columns.duplicated()].tolist()
    if repetidas:
        raise ValueError(f"Columnas con el mismo nombre en varios archivos: {repetidas}")
    faltantes = sorted(set(ids) - set(attrs.index))
    if faltantes:
        raise ValueError(f"Cuencas sin atributos: {faltantes}")

    attrs = attrs.loc[ids].copy()
    reporte = pd.read_csv(DIR_SERIES / "cleaning_report.csv", dtype={"gauge_id": str}).set_index("gauge_id")
    attrs["region"] = reporte["region"].reindex(attrs.index)
    for col in ["gauge_star", "gauge_end"]:
        attrs[col] = pd.to_datetime(attrs[col], format="%d/%m/%Y")
    return attrs


atributos_crudos = cargar_atributos(ids)
print(f"Atributos: {atributos_crudos.shape[0]} cuencas x {atributos_crudos.shape[1]} columnas")

Atributos: 248 cuencas x 75 columnas


### 1.4 Caudal faltante

Dentro del registro de cada estación, el caudal falta en el 6,1 % de los días. Casi todos son días sin medición (`no_data`); el resto son días anulados en el pre-filtrado por picos (`spike`) o valores congelados (`frozen`).

- **No se rellena.** El caudal es lo que el modelo aprende a predecir: rellenarlo haría que se entrenara y evaluara contra valores inventados. Los días faltantes quedan vacíos y NeuralHydrology los excluye de la pérdida y de las métricas. Sus forzantes siguen sirviendo como entrada para las ventanas vecinas.
- **No se descartan cuencas por tener vacíos.** El dataset ya exige 70 % de disponibilidad por estación. Un umbral de 5 % de vacíos eliminaría 136 de las 248 cuencas y reduciría la cobertura espacial.

En el código, el caudal se conserva solo en los días con estado `ok`; los demás quedan vacíos.

In [8]:
# El caudal se conserva solo en los días con estado "ok"; los demás quedan vacíos y no se imputan
for df in series.values():
    df.loc[df["streamflow_estado"] != "ok", "streamflow_m3s"] = np.nan

### 1.5 Forzantes: interpolación lineal de huecos de hasta 7 días

Las 4 forzantes (evapotraspiracion, temperatura maxima y minima, y precipitación) faltan los mismos días que el caudal. NeuralHydrology descarta toda ventana de entrenamiento con algún forzante vacío, y cada día vacío invalida las 365 ventanas que lo contienen: sin rellenar se perdería el 32 % de las ventanas.

- **Se interpolan solo los huecos internos de hasta 7 días** (con dato a ambos lados). Son el 83 % de los huecos, pero apenas el 10,5 % de los días vacíos (20.177 días).
- **Los huecos más largos quedan vacíos.** La interpolación lineal solo aproxima bien una variable que cambia poco dentro del hueco. Sobre varias semanas borraría la alternancia entre días secos (rachas de 2,7 días en mediana, 1.8) y lluviosos, que es la que explica los caudales bajos y altos. Además, ampliar el límite rinde cada vez menos: hasta 7 días se recuperan 17 ventanas por día interpolado; sin límite, menos de 1.
- **Los días rellenados quedan marcados** en la columna `forzantes_interpolados`, para revisar en la evaluación si los errores se concentran en ellos.

Las tablas de abajo muestran la duración de los huecos antes de interpolar y el efecto del límite sobre las ventanas de entrenamiento.

In [9]:
MAX_DIAS_INTERPOLACION = 7
forzantes = list(FORZANTES.values())


def rachas(evento):
    """Inicio y fin (exclusivo) de cada racha consecutiva de True."""
    bordes = np.flatnonzero(np.diff(np.r_[0, evento.astype(np.int8), 0]))
    return bordes[::2], bordes[1::2]


def interpolar_huecos_cortos(s, max_dias):
    """Interpola linealmente solo los huecos internos de hasta `max_dias` días.
    Devuelve la serie y una máscara con los días rellenados."""
    rellenar = np.zeros(len(s), dtype=bool)
    for ini, fin in zip(*rachas(s.isna().to_numpy())):
        if ini > 0 and fin < len(s) and fin - ini <= max_dias:
            rellenar[ini:fin] = True
    interpolada = s.interpolate(method="linear", limit_area="inside")
    return s.where(~rellenar, interpolada), rellenar


for df in series.values():
    if "forzantes_interpolados" in df:   # si la celda se vuelve a ejecutar, parte de los valores observados
        df.loc[df["forzantes_interpolados"], forzantes] = np.nan
    interpolado = np.zeros(len(df), dtype=bool)
    for col in forzantes:
        df[col], mascara = interpolar_huecos_cortos(df[col], MAX_DIAS_INTERPOLACION)
        interpolado |= mascara
    df["forzantes_interpolados"] = interpolado

print(f"Días con forzantes interpolados: {sum(int(df['forzantes_interpolados'].sum()) for df in series.values()):,}")

Días con forzantes interpolados: 20,177


In [10]:
# Duración de los huecos internos de las forzantes antes de interpolar (248 cuencas)
por_cuenca = {
    gid: ((df["pr"].isna() | df["forzantes_interpolados"]).to_numpy(),   # vacíos antes de interpolar
          df["streamflow_m3s"].notna().to_numpy(),
          EN_TRAIN)
    for gid, df in series.items()
}

duraciones = []
for vacio, _, _ in por_cuenca.values():
    ini, fin = rachas(vacio)
    interno = (ini > 0) & (fin < len(vacio))
    duraciones.extend((fin - ini)[interno])
duraciones = pd.Series(duraciones)

rangos = pd.cut(duraciones, [0, 1, 7, 30, 365, np.inf],
                labels=["1 día", "2 a 7 días", "8 a 30 días", "31 a 365 días", "más de 365 días"])
huecos = pd.DataFrame({"huecos": duraciones.groupby(rangos, observed=False).size(),
                       "días vacíos": duraciones.groupby(rangos, observed=False).sum()})
huecos["% de huecos"] = (100 * huecos["huecos"] / huecos["huecos"].sum()).round(1)
huecos["% de días vacíos"] = (100 * huecos["días vacíos"] / huecos["días vacíos"].sum()).round(1)
huecos

,huecos,días vacíos,% de huecos,% de días vacíos
1 día,7258,7258,53.2,3.8
2 a 7 días,4066,12919,29.8,6.7
8 a 30 días,1265,20326,9.3,10.6
31 a 365 días,972,101617,7.1,53.1
más de 365 días,74,49344,0.5,25.8


In [11]:
# Efecto del límite sobre las ventanas de entrenamiento: 365 días de forzantes completos
# que terminan en un día con caudal válido de 1981 a 2010
def evaluar_limite(limite):
    """Días interpolados y ventanas de entrenamiento válidas si se rellenan los huecos de hasta `limite` días."""
    dias_interpolados = ventanas = 0
    for vacio, q_valido, es_train in por_cuenca.values():
        ini, fin = rachas(vacio)
        rellenar = np.zeros(len(vacio), dtype=bool)
        for a, b in zip(ini, fin):
            if a > 0 and b < len(vacio) and b - a <= limite:
                rellenar[a:b] = True
        completo = (~vacio | rellenar).astype(int)
        ventana_completa = np.convolve(completo, np.ones(365, dtype=int))[: len(vacio)] == 365
        ventanas += int((ventana_completa & q_valido & es_train).sum())
        dias_interpolados += int(rellenar.sum())
    return dias_interpolados, ventanas

limites = {"sin interpolar": 0, "7 días": 7, "14 días": 14, "30 días": 30, "90 días": 90, "sin límite": np.inf}
compromiso = pd.DataFrame([evaluar_limite(l) for l in limites.values()], index=list(limites),
                          columns=["días interpolados", "ventanas de entrenamiento"])
compromiso["% de días vacíos interpolados"] = (100 * compromiso["días interpolados"] / duraciones.sum()).round(1)
compromiso["% de ventanas"] = (100 * compromiso["ventanas de entrenamiento"]
                               / compromiso["ventanas de entrenamiento"].max()).round(1)
compromiso["ventanas ganadas por día interpolado"] = (compromiso["ventanas de entrenamiento"].diff()
                                                      / compromiso["días interpolados"].diff()).round(1)
compromiso

,días interpolados,ventanas de entrenamiento,% de días vacíos interpolados,% de ventanas,ventanas ganadas por día interpolado
sin interpolar,0,1473247,0.0,67.6,NaN
7 días,20177,1819212,10.5,83.4,17.1
14 días,26975,1883762,14.1,86.4,9.5
30 días,40503,1976309,21.2,90.6,6.8
90 días,71761,2097075,37.5,96.2,3.9
sin límite,191464,2180431,100.0,100.0,0.7


### 1.6 Caudal en mm/día

El caudal viene en m³/s, una magnitud que crece con el tamaño de la cuenca. Se convierte a lámina de agua sobre la cuenca, la misma unidad de la precipitación y la evapotranspiración:

$$Q_{\text{mm/día}} = Q_{\text{m}^3\text{/s}} \times \frac{86{,}4}{\text{área}_{\text{km}^2}}$$

En mm/día la variación entre cuencas baja a un factor de 122 y el modelo aprende la respuesta de la cuenca, no su tamaño. Además, NeuralHydrology estandariza el caudal con una sola media y una sola desviación para todas las cuencas: en m³/s las dominarían los ríos grandes, y el caudal estandarizado de la cuenca mediana casi no variaría (desviación de 0,02, frente a 0,55 en mm/día). La media convertida coincide con el `q_mean` publicado en el dataset.

In [12]:
def caudal_a_mm(q_m3s, area_km2):
    """m³/s × 86 400 s/día × 1 000 mm/m ÷ (área × 10⁶ m²/km²) = m³/s × 86,4 / área."""
    return q_m3s * 86.4 / area_km2


for gid, df in series.items():
    df[OBJETIVO] = caudal_a_mm(df["streamflow_m3s"], atributos_crudos.at[gid, "area"])

In [13]:
# Caudal medio por cuenca en cada unidad (período de entrenamiento)
q = pd.DataFrame({gid: {"media_m3s": df.loc[EN_TRAIN, "streamflow_m3s"].mean(),
                        "media_mm": df.loc[EN_TRAIN, "streamflow"].mean(),
                        "desv_m3s": df.loc[EN_TRAIN, "streamflow_m3s"].std(),
                        "desv_mm": df.loc[EN_TRAIN, "streamflow"].std()}
                  for gid, df in series.items()}).T
desv_global_m3s, desv_global_mm = pd.concat([df.loc[EN_TRAIN, ["streamflow_m3s", "streamflow"]]
                                             for df in series.values()]).std()

media_total = pd.Series({gid: df["streamflow"].mean() for gid, df in series.items()})
razon = media_total / atributos_crudos["q_mean"]
print(f"Verificación: media convertida / q_mean publicado = {razon.median():.3f} (mediana entre cuencas)\n")

pd.DataFrame({
    "m³/s": [q["media_m3s"].min(), q["media_m3s"].max(), q["media_m3s"].max() / q["media_m3s"].min(),
             (q["desv_m3s"] / desv_global_m3s).median()],
    "mm/día": [q["media_mm"].min(), q["media_mm"].max(), q["media_mm"].max() / q["media_mm"].min(),
               (q["desv_mm"] / desv_global_mm).median()],
}, index=["caudal medio mínimo", "caudal medio máximo", "razón máximo / mínimo",
          "desviación del caudal estandarizado (cuenca mediana)"]).astype(float).round(3)

Verificación: media convertida / q_mean publicado = 1.000 (mediana entre cuencas)



,m³/s,mm/día
caudal medio mínimo,0.174,0.175
caudal medio máximo,7401.838,21.357
razón máximo / mínimo,42528.447,122.171
desviación del caudal estandarizado (cuenca mediana),0.024,0.551


### 1.7 Re-escalamiento y filtrado de atributos estáticos

- **Área.** Va de 25 a 260 372 km² y su distribución es muy asimétrica: estandarizada directamente, el 88 % de las cuencas quedaría entre −0,5 y +0,5 desviaciones y el modelo apenas distinguiría su tamaño. Entra como `log10(area)`, que reduce la asimetría de 5,9 a 0,5. El área original se conserva para convertir el caudal (1.6).
- **`tc_representative`.** Es un error del dataset: sus valores salen de las fórmulas de tiempo de concentración, pero están asignados a cuencas equivocadas. Solo 3 de 346 coinciden con una fórmula de su propia cuenca, y su correlación con la longitud del cauce es nula (−0,03, frente a 0,97 de `tc_Johnstone`). En la lista de candidatos (1.1) se reemplaza por `tc_Johnstone`, la fórmula de la que provienen más valores (211 de 346).
- **Atributos casi vacíos.** Un atributo de porcentaje (cobertura, geología, suelos o capacidad de uso) se descarta si menos del 5 % de las cuencas tiene más del 1 % de su área en esa clase. Sin contraste entre cuencas el modelo no puede aprender su efecto, y un valor distinto de cero solo identificaría a una cuenca. Se descartan `class_1`, `aridisols_perc` y `non_identi_land_perc`. Por el mismo criterio, `spodosols_perc`, `coal_mine_pit_perc` y `perpetual_snow_perc` ya se habían excluido de la lista de candidatos.

In [14]:
# Área: escala logarítmica
atributos = atributos_crudos.assign(log_area=np.log10(atributos_crudos["area"]))

# tc_representative no se usa: en la lista de candidatos (1.1) se reemplazó por tc_Johnstone

# Atributos de porcentaje casi vacíos
UMBRAL_PCT = 1.0          # % del área desde el que se considera que una clase está presente en la cuenca
MIN_FRAC_CUENCAS = 0.05   # fracción mínima de cuencas en las que debe estar presente
candidatos_pct = [c for g in ["cobertura", "geologia", "suelos", "capacidad_uso"] for c in ATRIBUTOS_CANDIDATOS[g]]
presencia = (atributos[candidatos_pct] > UMBRAL_PCT).mean()
descartados = presencia[presencia < MIN_FRAC_CUENCAS].index.tolist()
print("Atributos descartados por estar casi vacíos:", descartados)

Atributos descartados por estar casi vacíos: ['non_identi_land_perc', 'aridisols_perc', 'class_1']


In [15]:
# Área: distribución antes y después del logaritmo
area = atributos_crudos["area"]
z_area = (area - area.mean()) / area.std()
print(f"Área: {area.min():,.0f} a {area.max():,.0f} km² | asimetría {area.skew():.1f}, con log10 "
      f"{np.log10(area).skew():.1f} | sin log, {100 * (z_area.abs() < 0.5).mean():.0f} % de las cuencas "
      f"entre ±0,5 desviaciones")

# tc_representative: ¿coincide con alguna fórmula de su propia cuenca? (346 cuencas del dataset)
fisiograficos = leer_csv(DIR_DATOS / "10_CAMELS_COL_Physiograpic_characteristics.csv")
formulas_tc = ["tc_kirpich", "tc_chow", "tc_Johnstone", "tc_Engi_Corps"]
en_su_cuenca = (np.abs(fisiograficos[formulas_tc].to_numpy()
                       - fisiograficos[["tc_representative"]].to_numpy()) < 0.01).any(axis=1)

def formula_de_origen(valor):
    """Fórmula cuyo valor, en cualquier cuenca, coincide con `valor`."""
    coinciden = [f for f in formulas_tc if (np.abs(fisiograficos[f] - valor) < 0.01).any()]
    return coinciden[0] if len(coinciden) == 1 else ("ninguna" if not coinciden else "varias")

print(f"\ntc_representative coincide con una fórmula de su propia cuenca: {en_su_cuenca.sum()} de {len(fisiograficos)}")
print("Fórmula de la que proviene cada valor (en cualquier cuenca):",
      pd.Series([formula_de_origen(v) for v in fisiograficos["tc_representative"]]).value_counts().to_dict())
print(f"Spearman con la longitud del cauce: tc_representative "
      f"{atributos_crudos['tc_representative'].corr(atributos_crudos['streng_chanel'], method='spearman'):.2f}, "
      f"tc_Johnstone {atributos_crudos['tc_Johnstone'].corr(atributos_crudos['streng_chanel'], method='spearman'):.2f}")

# Atributos casi vacíos: cuencas con más de 1 % de su área en la clase (los 5 menos frecuentes)
print("\nCuencas con más de 1 % del área:", (atributos[candidatos_pct] > UMBRAL_PCT).sum().nsmallest(5).to_dict())

Área: 25 a 260,372 km² | asimetría 5.9, con log10 0.5 | sin log, 88 % de las cuencas entre ±0,5 desviaciones

tc_representative coincide con una fórmula de su propia cuenca: 3 de 346


Fórmula de la que proviene cada valor (en cualquier cuenca): {'tc_Johnstone': 211, 'tc_Engi_Corps': 73, 'varias': 57, 'tc_chow': 5}


Spearman con la longitud del cauce: tc_representative -0.03, tc_Johnstone 0.97

Cuencas con más de 1 % del área: {'class_1': 0, 'non_identi_land_perc': 1, 'aridisols_perc': 1, 'histosols_perc': 14, 'eroded_misce_perc': 16}


### 1.8 Índices climáticos estáticos recalculados

Los índices climáticos resumen el clima de cada cuenca en valores fijos:

| Índice | Definición |
|---|---|
| `p_mean`, `pet_mean` | Precipitación y evapotranspiración potencial medias (mm/día) |
| `aridity` | `pet_mean / p_mean` |
| `high_prec_freq`, `high_prec_dur` | Días por año con `pr ≥ 5 × p_mean`, y duración media de sus rachas |
| `low_prec_freq`, `low_prec_dur` | Días por año con `pr < 1 mm`, y duración media de sus rachas |

Los índices publicados en CAMELS-COL no se usan por dos razones. Se calcularon con todo el registro, hasta 2022, que incluye el período de prueba. Y usan la evapotranspiración anterior a su corrección (media de 8,9 frente a 3,3 mm/día), con la que la aridez mediana sería 1,62, propia de cuencas secas, en lugar de 0,59, propia de cuencas húmedas. Además, el `low_prec_dur` publicado (mediana de 16,4 días) no se pudo reproducir con su definición.

Por eso se recalculan con los forzantes observados de 1981 a 2010, sin días interpolados, y se agregan `p_mean` y `pet_mean`, que no vienen en el dataset. Los días vacíos cortan las rachas. Como solo usan forzantes, existen también en cuencas no aforadas.

In [16]:
def duracion_media(evento):
    """Duración media en días de las rachas consecutivas de True (0 si no hay)."""
    ini, fin = rachas(evento)
    return float((fin - ini).mean()) if len(ini) else 0.0


def indices_climaticos(pr, pet):
    """Índices climáticos con las definiciones de CAMELS-COL. Reciben solo valores
    observados; los días vacíos cortan las rachas."""
    anios = pr.notna().sum() / 365.25
    p_mean, pet_mean = pr.mean(), pet.mean()
    alta = (pr >= 5 * p_mean).to_numpy()   # NaN >= x es False: corta la racha
    seca = (pr < 1).to_numpy()
    return {"p_mean": p_mean, "pet_mean": pet_mean, "aridity": pet_mean / p_mean,
            "high_prec_freq": alta.sum() / anios, "high_prec_dur": duracion_media(alta),
            "low_prec_freq": seca.sum() / anios, "low_prec_dur": duracion_media(seca)}


climaticos = {}
for gid, df in series.items():
    train = df[EN_TRAIN]
    observado = ~train["forzantes_interpolados"]   # sin los días interpolados en 1.5
    climaticos[gid] = indices_climaticos(train["pr"].where(observado), train["poten_evapo"].where(observado))
atributos = atributos.assign(**pd.DataFrame.from_dict(climaticos, orient="index"))

In [17]:
# Índices publicados frente a recalculados: medianas entre cuencas y Spearman (orden de las cuencas)
originales = leer_csv(DIR_DATOS / "08_CAMELS_COL_Climatic_indices.csv").loc[ids]
indices = ["aridity", "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur"]
pd.DataFrame({
    "mediana original (1981–2022)": originales[indices].median(),
    "mediana recalculada (1981–2010)": atributos[indices].median(),
    "Spearman original vs recalculado": [atributos[c].corr(originales[c], method="spearman") for c in indices],
}).round(2)

,mediana original (1981–2022),mediana recalculada (1981–2010),Spearman original vs recalculado
aridity,1.62,0.59,1.00
high_prec_freq,9.67,9.84,0.99
high_prec_dur,1.36,1.04,0.88
low_prec_freq,157.94,163.79,0.99
low_prec_dur,16.43,2.65,0.87


### 1.9 Partición temporal

| Período | Años | Uso |
|---|---|---|
| Entrenamiento | 1981 a 2010 (30) | Ajuste de los pesos. Cubre varios ciclos de El Niño y La Niña |
| Validación | 2011 a 2015 (5) | Parada temprana y elección de `hidden_size` |
| Prueba | 2016 a 2022 (7) | Evaluación final, en los años más recientes (Yu & Tolson, 2026) |

La partición es cronológica, no aleatoria, porque el caudal diario está autocorrelacionado: con días al azar, cada día de prueba tendría a sus vecinos en el entrenamiento, y la métrica mediría la capacidad de interpolar entre días conocidos, no la de simular años no vistos.

No todas las cuencas tienen caudal en cada período: 248 en entrenamiento, 213 en validación y 189 en prueba. Las que no tienen datos de prueba se concentran en Caribe (39 %) y Amazonas (36 %), así que la evaluación temporal queda inclinada hacia Magdalena-Cauca; esto se declarará al reportar los resultados.

In [18]:
def asignar_periodo(fechas):
    """Etiqueta cada fecha con su período: train, validation o test."""
    etiqueta = np.full(len(fechas), "", dtype=object)
    for nombre, (ini, fin) in PERIODOS.items():
        etiqueta[(fechas >= ini) & (fechas <= fin)] = nombre
    return pd.Categorical(etiqueta, categories=list(PERIODOS))


for df in series.values():
    df["periodo"] = asignar_periodo(df.index)

In [19]:
# Cuencas con caudal en cada período y, por región, cuántas no tienen datos de prueba
dias_por_periodo = pd.DataFrame({
    "region": atributos["region"],
    **{f"dias_q_{p}": pd.Series({gid: int((df[OBJETIVO].notna() & (df["periodo"] == p)).sum())
                                 for gid, df in series.items()}) for p in PERIODOS},
})
print("Cuencas con caudal válido por período:",
      {p: int((dias_por_periodo[f"dias_q_{p}"] > 0).sum()) for p in PERIODOS}, "\n")

por_region = dias_por_periodo.groupby("region").agg(
    cuencas=("dias_q_test", "size"),
    con_caudal_en_test=("dias_q_test", lambda s: int((s > 0).sum())),
)
por_region.loc["Total"] = [len(dias_por_periodo), int((dias_por_periodo["dias_q_test"] > 0).sum())]
por_region = por_region.astype(int)
por_region["% sin caudal en test"] = (100 * (1 - por_region["con_caudal_en_test"] / por_region["cuencas"])).round(0)
por_region

Cuencas con caudal válido por período: {'train': 248, 'validation': 213, 'test': 189} 



,cuencas,con_caudal_en_test,% sin caudal en test
region,,,
Amazon,11,7,36.0
Caribbean,44,27,39.0
Magdalena-Cauca,136,108,21.0
Orinoco,37,31,16.0
Pacific,20,16,20.0
Total,248,189,24.0


### 1.10 Estandarización

El preprocesamiento no estandariza ninguna variable: los datos quedan en sus unidades físicas. NeuralHydrology calcula la media y la desviación de cada variable al entrenar, solo con las cuencas y el período de entrenamiento de esa corrida, y las guarda en `train_data/train_data_scaler.yml`.

Calcularlas con todo el dataset incluiría el período de prueba y las cuencas retenidas en la validación espacial, es decir, información de los datos con los que se evalúa. Además, en la validación espacial cada partición tiene otras cuencas de entrenamiento y los estadísticos deben recalcularse. Por eso este paso no tiene código propio.

### 1.11 Tabla final

Se verifica que los atributos del modelo existan y no tengan vacíos, y se unen las series de las 248 cuencas con sus atributos estáticos en una sola tabla, con una fila por cuenca y día. La base nueva se escribe en `data/processed/`:

| Archivo | Contenido |
|---|---|
| `camels_col_diario.parquet` | Tabla única: forzantes, caudal (m³/s y mm/día), estado del caudal, marca de interpolación, período y los 48 atributos estáticos |
| `atributos.csv` | Una fila por cuenca: atributos del modelo, auxiliares y firmas hidrológicas |
| `resumen_cuencas.csv` | Control de calidad por cuenca: caudal válido, días interpolados y días con caudal por período |

In [20]:
atributos_modelo = [c for cols in ATRIBUTOS_CANDIDATOS.values() for c in cols if c not in descartados]
faltan = [c for c in atributos_modelo if c not in atributos.columns]
assert not faltan, f"Atributos del modelo que no existen: {faltan}"
con_vacios = atributos[atributos_modelo].columns[atributos[atributos_modelo].isna().any()].tolist()
assert not con_vacios, f"Atributos del modelo con vacíos: {con_vacios}"

finales = {"forzantes": forzantes, "objetivo": OBJETIVO, "atributos_estaticos": atributos_modelo}

# Variables finales del modelo, por grupo
grupos = {"forzantes": forzantes,
          **{g: [c for c in cols if c not in descartados] for g, cols in ATRIBUTOS_CANDIDATOS.items()},
          "objetivo": [OBJETIVO]}
filas = [f"| {g} | {len(v)} | {', '.join(f'`{c}`' for c in v)} |" for g, v in grupos.items()]
display(Markdown("| Grupo | n | Variables |\n|---|---|---|\n" + "\n".join(filas)))

| Grupo | n | Variables |
|---|---|---|
| forzantes | 4 | `pr`, `poten_evapo`, `t_min`, `t_max` |
| fisiograficos | 10 | `log_area`, `gravelius_index`, `factor_form`, `maximum_ele`, `mean_ele`, `minimum_ele`, `streng_chanel`, `equi_slope`, `cn_catchment`, `tc_Johnstone` |
| climaticos | 7 | `p_mean`, `pet_mean`, `aridity`, `high_prec_freq`, `high_prec_dur`, `low_prec_freq`, `low_prec_dur` |
| cobertura | 5 | `forest_perc`, `nat_non_forest_form_perc`, `agricul_livestock_perc`, `non_vegeted_perc`, `water_bodies_perc_cob` |
| geologia | 6 | `unconso_depo_perc`, `hypabyssal_rock_perc`, `metamor_rock_perc`, `plutonic_rock_perc`, `sedimen_rock_perc`, `volcanic_rock_perc` |
| suelos | 13 | `inceptisols_perc`, `entisols_perc`, `andisols_perc`, `alfisols_perc`, `vertisols_perc`, `ultisols_perc`, `oxisols_perc`, `mollisols_perc`, `histosols_perc`, `urban_perc`, `eroded_misce_perc`, `rocky_misce_perc`, `water_bodies_perc_suelo` |
| capacidad_uso | 7 | `class_2`, `class_3`, `class_4`, `class_5`, `class_6`, `class_7`, `class_8` |
| objetivo | 1 | `streamflow` |

In [21]:
# Tabla única: series de todas las cuencas + atributos estáticos del modelo
columnas = ["date", "periodo", *forzantes, "streamflow_m3s", "streamflow_estado", "forzantes_interpolados", OBJETIVO]
diario = pd.concat([df.reset_index()[columnas].assign(gauge_id=gid) for gid, df in series.items()], ignore_index=True)
diario = diario[["gauge_id", *columnas]].merge(atributos[atributos_modelo].astype("float32"), left_on="gauge_id",
                                               right_index=True, how="left", validate="many_to_one")
flotantes = diario.select_dtypes("float64").columns
diario[flotantes] = diario[flotantes].astype("float32")
diario["gauge_id"] = diario["gauge_id"].astype("category")
diario["streamflow_estado"] = diario["streamflow_estado"].astype("category")

# Control de calidad por cuenca
en_registro = {gid: df["streamflow_estado"] != "sin_registro" for gid, df in series.items()}
resumen = pd.DataFrame([{
    "gauge_id": gid,
    "region": atributos.at[gid, "region"],
    "inicio_registro": df.index[en_registro[gid].to_numpy()].min().date(),
    "fin_registro": df.index[en_registro[gid].to_numpy()].max().date(),
    "q_valido_pct_registro": round(100 * df.loc[en_registro[gid], OBJETIVO].notna().mean(), 2),
    **{f"dias_q_{p}": int(dias_por_periodo.at[gid, f"dias_q_{p}"]) for p in PERIODOS},
    "dias_forzantes_interpolados": int(df["forzantes_interpolados"].sum()),
    "dias_forzantes_vacios_tras_interpolar": int(df.loc[en_registro[gid], "pr"].isna().sum()),
    "anios_forzantes_train": round(int((df.loc[EN_TRAIN, "pr"].notna()
                                        & ~df.loc[EN_TRAIN, "forzantes_interpolados"]).sum()) / 365.25, 1),
} for gid, df in series.items()])

DIR_SALIDA.mkdir(parents=True, exist_ok=True)
diario.to_parquet(DIR_SALIDA / "camels_col_diario.parquet", index=False)
atributos[AUXILIARES + atributos_modelo + FIRMAS_HIDROLOGICAS].rename_axis("gauge_id").to_csv(DIR_SALIDA / "atributos.csv")
resumen.to_csv(DIR_SALIDA / "resumen_cuencas.csv", index=False)

print(f"Tabla diaria: {len(diario):,} filas x {diario.shape[1]} columnas ({len(series)} cuencas x {len(CALENDARIO):,} días)")
print(f"Días con caudal válido: {int(diario[OBJETIVO].notna().sum()):,} | "
      f"días con forzantes interpolados: {int(diario['forzantes_interpolados'].sum()):,}")
print(f"Archivos escritos en {DIR_SALIDA.relative_to(ROOT)}/")

Tabla diaria: 3,804,320 filas x 59 columnas (248 cuencas x 15,340 días)
Días con caudal válido: 2,976,524 | días con forzantes interpolados: 20,177
Archivos escritos en data\processed/


### 1.12 Formato NeuralHydrology

NeuralHydrology no trae CAMELS-COL, así que se usa su lector genérico (`dataset: generic`), que exige esta estructura:

```
data/processed/
├── time_series/
│   ├── 11027010.nc            ← una serie por cuenca, coordenada `date`, vacíos como NaN
│   └── ...
└── attributes/
    └── atributos_modelo.csv   ← primera columna = id de la cuenca (se lee como texto)
```

- Los nombres de las variables en los `.nc` deben coincidir exactamente con `dynamic_inputs` y `target_variables` de la configuración.
- Solo se guardan las 4 forzantes y el caudal en mm/día.
- Los atributos no pueden tener vacíos (se verifica en 1.11).

In [22]:
# import sys
# !{sys.executable} -m pip install xarray

In [23]:
import xarray as xr

DIR_NH = DIR_SALIDA
(DIR_NH / "time_series").mkdir(parents=True, exist_ok=True)
(DIR_NH / "attributes").mkdir(parents=True, exist_ok=True)

variables_nc = finales["forzantes"] + [finales["objetivo"]]
for gid, g in diario.groupby("gauge_id", observed=True):
    xr.Dataset.from_dataframe(g.set_index("date")[variables_nc]).to_netcdf(DIR_NH / "time_series" / f"{gid}.nc")

atributos[finales["atributos_estaticos"]].rename_axis("gauge_id").to_csv(DIR_NH / "attributes" / "atributos_modelo.csv")

print(f"{len(list((DIR_NH / 'time_series').glob('*.nc')))} archivos .nc escritos\n")
with xr.open_dataset(DIR_NH / "time_series" / f"{ids[0]}.nc") as ds:
    print(ds)

248 archivos .nc escritos



<xarray.Dataset> Size: 430kB
Dimensions:      (date: 15340)
Coordinates:
  * date         (date) datetime64[ns] 123kB 1981-01-01 ... 2022-12-31
Data variables:
    pr           (date) float32 61kB ...
    poten_evapo  (date) float32 61kB ...
    t_min        (date) float32 61kB ...
    t_max        (date) float32 61kB ...
    streamflow   (date) float32 61kB ...


## 2. Forzantes adicionales: ERA5-Land

**Por qué.** Las 4 forzantes de CAMELS-COL tienen un límite: la lluvia viene de una sola fuente (CHIRPS) y no hay humedad del suelo, radiación ni humedad del aire. ERA5-Land (ECMWF, Copernicus) es un reanálisis de superficie en celdas de 0,1° (unos 11 km) que sí trae esas variables, con errores distintos a los de CHIRPS. Es también la fuente de las forzantes de Caravan.

En el modelado (secciones 8.4 a 8.7) sube el NSE mediano de prueba de 0,410 a 0,437.

**Qué produce esta sección:**
- `data/externos/era5land/era5land_cuencas_diario.parquet`: 15 variables diarias por cuenca, de 1981 a 2022, con las mismas filas y el mismo orden que `camels_col_diario.parquet`;
- `data/processed_era5land/`: copia de la base de 1.12 con las 15 variables agregadas a cada serie, para el LSTM.

**Regla del proyecto:** la escorrentía de ERA5-Land es la salida del modelo de suelo, calculada solo con la meteorología del reanálisis. No usa caudal observado, así que puede ser entrada. El caudal observado solo aparece en 2.4, para evaluar.

| Variable | Origen en ERA5-Land | Cómo se lleva al día local | Unidad |
|---|---|---|---|
| `era5l_pr_mm` | `tp` (precipitación total) | Acumulada: total de 19:00 del día anterior a 19:00 | mm/día |
| `era5l_srad_wm2`, `era5l_lrad_wm2` | `ssrd`, `strd` (radiación solar y térmica descendente) | Acumulada, dividida por 86 400 s | W/m² |
| `era5l_evap_mm` | `e` (evaporación total, con signo cambiado) | Acumulada | mm/día |
| `era5l_esc_total_mm`, `era5l_esc_sup_mm`, `era5l_esc_sub_mm` | `ro`, `sro`, `ssro` (escorrentía del modelo de suelo) | Acumulada | mm/día |
| `era5l_suelo1` a `era5l_suelo4` | `swvl1` a `swvl4` (humedad del suelo: 0–7, 7–28, 28–100 y 100–289 cm) | Valor de las 07:00 | m³/m³ |
| `era5l_rocio_c` | `d2m` (temperatura de rocío a 2 m) | Promedio de 01, 07, 13 y 19 h | °C |
| `era5l_pvapor_kpa` | Desde `d2m` (FAO-56, ec. 14), en cada celda y hora | Promedio de 01, 07, 13 y 19 h | kPa |
| `era5l_viento_ms` | Rapidez desde `u10` y `v10`, en cada celda y hora | Promedio de 01, 07, 13 y 19 h | m/s |
| `era5l_presion_kpa` | `sp` (presión en superficie) | Promedio de 01, 07, 13 y 19 h | kPa |

### 2.1 Descarga desde Copernicus CDS

**Acceso:**
- Hace falta una cuenta gratuita del Climate Data Store y aceptar la licencia de ERA5-Land.
- La clave va en el archivo `.cdsapirc` de la carpeta del usuario, con dos líneas: `url: https://cds.climate.copernicus.eu/api` y `key: <clave>`. **Nunca en el notebook ni en git.**
- Requiere `pip install cdsapi`.

**Qué se pide:**
- La caja que contiene las 248 cuencas con margen: latitud de −1,7 a 11,4 y longitud de −78,1 a −67,6.
- Tres tipos de pedido por año, de 1981 a 2022:
  - **acumuladas:** 7 variables a las 00 UTC. El valor de las 00 UTC del día D + 1 es el total del día UTC D, así que se baja también el 1/1/2023;
  - **suelo:** las 4 capas a las 12 UTC;
  - **instantáneas:** rocío, viento y presión a las 00, 06, 12 y 18 UTC.

**Cómo corre:**
- 126 pedidos (más el día extra), 3 a la vez.
- Se reanuda sola: salta los archivos completos, y cada archivo se escribe como `.part` y se renombra al terminar.
- Mantiene el equipo despierto mientras corre.

**Costo:** tardó 31,5 horas (del 17/11 a las 17:41 al 19/11 a la 01:10), por la cola del CDS, y ocupa unos 11 GB. Para descargar, cambia `EJECUTAR_DESCARGA_ERA5L` a `True`. El avance queda en `data/externos/era5land/descarga.log`.

In [ ]:
import ctypes
import logging
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

EJECUTAR_DESCARGA_ERA5L = False
DIR_ERA5L = ROOT / "data" / "externos" / "era5land"
CAJA_ERA5L = [11.4, -78.1, -1.7, -67.6]                 # norte, oeste, sur, este
ANIOS_ERA5L = range(1981, 2023)
PEDIDOS_ERA5L = {
    "acumuladas": (["total_precipitation", "surface_solar_radiation_downwards", "surface_thermal_radiation_downwards",
                    "total_evaporation", "runoff", "surface_runoff", "sub_surface_runoff"], ["00:00"]),
    "suelo": ([f"volumetric_soil_water_layer_{i}" for i in range(1, 5)], ["12:00"]),
    "instantaneas": (["2m_dewpoint_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind",
                      "surface_pressure"], ["00:00", "06:00", "12:00", "18:00"]),
}
HILOS_DESCARGA = 3
MESES_ERA5L, DIAS_ERA5L = [f"{m:02d}" for m in range(1, 13)], [f"{d:02d}" for d in range(1, 32)]
DEMASIADO_GRANDE = ("too large", "cost limit", "limits exceeded", "request too")


def descargar_era5land():
    import cdsapi   # la clave se lee de ~/.cdsapirc

    for t in PEDIDOS_ERA5L:
        (DIR_ERA5L / t).mkdir(parents=True, exist_ok=True)
    registro = logging.getLogger("era5land")
    registro.setLevel(logging.INFO)
    if not registro.handlers:
        h = logging.FileHandler(DIR_ERA5L / "descarga.log", encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(threadName)s %(message)s"))
        registro.addHandler(h)
    local = threading.local()

    def pedir(tipo, anio, meses, dias, destino):
        if not hasattr(local, "c"):
            local.c = cdsapi.Client(quiet=True, progress=False, sleep_max=60, timeout=600)
        variables, horas = PEDIDOS_ERA5L[tipo]
        parte, t0 = destino.with_suffix(".part"), time.time()
        local.c.retrieve("reanalysis-era5-land", {"variable": variables, "year": str(anio), "month": meses, "day": dias,
                                                  "time": horas, "area": CAJA_ERA5L, "data_format": "netcdf",
                                                  "download_format": "unarchived"}, str(parte))
        os.replace(parte, destino)
        return time.time() - t0, destino.stat().st_size / 1e6

    def con_reintentos(tipo, anio, meses, dias, destino, intentos=4):
        for k in range(1, intentos + 1):
            try:
                return pedir(tipo, anio, meses, dias, destino)
            except Exception as e:   # noqa: BLE001
                if any(s in str(e).lower() for s in DEMASIADO_GRANDE):
                    raise
                registro.warning("Fallo %s %s (intento %d): %s", tipo, anio, k, str(e)[:300])
                time.sleep(60 * k)
        raise RuntimeError(f"Sin éxito tras {intentos} intentos: {tipo} {anio}")

    def trabajo(tipo, anio):
        destino = DIR_ERA5L / tipo / f"{tipo}_{anio}.nc"
        mensuales = [DIR_ERA5L / tipo / f"{tipo}_{anio}_{m}.nc" for m in MESES_ERA5L]
        if destino.exists() or all(p.exists() for p in mensuales):
            return 0, 0
        try:
            return con_reintentos(tipo, anio, MESES_ERA5L, DIAS_ERA5L, destino)
        except Exception as e:   # noqa: BLE001
            if not any(s in str(e).lower() for s in DEMASIADO_GRANDE):
                raise
        registro.info("%s %s: pedido anual demasiado grande, se parte en meses", tipo, anio)
        totales = [con_reintentos(tipo, anio, [m], DIAS_ERA5L, p) for m, p in zip(MESES_ERA5L, mensuales) if not p.exists()]
        return sum(x[0] for x in totales), sum(x[1] for x in totales)

    if sys.platform == "win32":   # evita que el equipo se suspenda mientras descarga
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)
    t0, trabajos = time.time(), [(t, a) for a in ANIOS_ERA5L for t in PEDIDOS_ERA5L]
    try:
        extra = DIR_ERA5L / "acumuladas" / "acumuladas_2023_01_01.nc"   # cierra el total del 31/12/2022
        if not extra.exists():
            con_reintentos("acumuladas", 2023, ["01"], ["01"], extra)
        hechos, fallos = 0, []
        with ThreadPoolExecutor(max_workers=HILOS_DESCARGA, thread_name_prefix="h") as ex:
            futuros = {ex.submit(trabajo, t, a): (t, a) for t, a in trabajos}
            for f in as_completed(futuros):
                t, a = futuros[f]
                try:
                    seg, mb = f.result()
                    hechos += 1
                    registro.info("[%d/%d] %s %d: %.0f s, %.1f MB | %.1f h", hechos, len(trabajos), t, a, seg, mb,
                                  (time.time() - t0) / 3600)
                except Exception as e:   # noqa: BLE001
                    fallos.append((t, a))
                    registro.error("FALLÓ %s %d: %s", t, a, str(e)[:300])
        registro.info("Fin en %.1f h. Completos: %d. Fallidos: %s", (time.time() - t0) / 3600, hechos, fallos or "ninguno")
    finally:
        if sys.platform == "win32":
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)


if EJECUTAR_DESCARGA_ERA5L:
    descargar_era5land()

estado_descarga = pd.DataFrame({t: {"archivos": len(list((DIR_ERA5L / t).glob(f"{t}_*.nc"))),
                                    "GB": round(sum(p.stat().st_size for p in (DIR_ERA5L / t).glob(f"{t}_*.nc")) / 1e9, 2)}
                                for t in PEDIDOS_ERA5L}).T
faltan_era5l = [f"{t} {a}" for t in PEDIDOS_ERA5L for a in ANIOS_ERA5L if not (DIR_ERA5L / t / f"{t}_{a}.nc").exists()]
display(estado_descarga)
print("Pedidos que faltan:", faltan_era5l or "ninguno",
      "| día extra 1/1/2023:", (DIR_ERA5L / "acumuladas" / "acumuladas_2023_01_01.nc").exists())

,archivos,GB
acumuladas,43.0,3.76
suelo,42.0,1.42
instantaneas,42.0,6.84


Pedidos que faltan: ninguno | día extra 1/1/2023: True


### 2.2 Peso de cada celda en cada cuenca

ERA5-Land es una grilla de celdas sin identificador de cuenca, así que la unión es espacial: el valor de una cuenca es el promedio de las celdas que toca, ponderado por el área de cada celda que queda dentro.
- **Fracción dentro:** se mide con 20 × 20 puntos por celda y la regla par-impar sobre todos los anillos del polígono, así que los huecos restan.
  - Se calcula con un barrido por filas, que da lo mismo que `contains_points` punto a punto (2.4) y hace las 248 cuencas en segundos.
- **Peso:** fracción × área de la celda sobre el elipsoide WGS84, renormalizado sobre las celdas de tierra (las de mar no tienen datos en ERA5-Land).
- **Polígonos:** los de CAMELS-COL vienen en WGS 84 World Mercator (EPSG:3395, elipsoidal) y se pasan a grados con la fórmula elipsoidal.
  - Con la fórmula esférica (Web Mercator) quedarían corridos hacia el ecuador entre 1,5 y 8 km.

La celda lee los polígonos de `data/raw/03_CAMELS_COL_Basin_boundary/` y la grilla del primer archivo descargado. Para recalcular, cambia `EJECUTAR_PESOS_ERA5L` a `True`; tarda unos segundos.

In [ ]:
import struct

import netCDF4

EJECUTAR_PESOS_ERA5L = False
DIR_LIMITES = ROOT / "data" / "raw" / "03_CAMELS_COL_Basin_boundary" / "03_CAMELS_COL_Basin_boundary"
PESOS_ERA5L = DIR_ERA5L / "pesos_cuencas.csv"
RESUMEN_PESOS_ERA5L = DIR_ERA5L / "pesos_resumen.csv"
PASO_ERA5L, SUB_CELDA = 0.1, 20                      # grados por celda; puntos por lado para medir la fracción
IDS_ERA5L = sorted(p.stem for p in (DIR_SALIDA / "time_series").glob("*.nc"))   # las cuencas de 1.12
R_WGS84, EXCENTRICIDAD_WGS84 = 6378137.0, 0.0818191908426


def leer_shp(ruta):
    """Lector mínimo de shapefile de polígonos: una lista de anillos (n × 2) por registro."""
    b = Path(ruta).read_bytes()
    formas, pos = [], 100
    while pos < len(b):
        largo = int.from_bytes(b[pos + 4:pos + 8], "big") * 2
        registro = b[pos + 8:pos + 8 + largo]
        n_anillos, n_puntos = np.frombuffer(registro, "<i4", 2, 36)
        inicios = np.frombuffer(registro, "<i4", n_anillos, 44)
        xy = np.frombuffer(registro, "<f8", 2 * n_puntos, 44 + 4 * n_anillos).reshape(-1, 2)
        formas.append(np.split(xy, inicios[1:]))
        pos += 8 + largo
    return formas


def leer_ids_dbf(ruta):
    b = Path(ruta).read_bytes()
    n, largo_encabezado, largo_registro = struct.unpack("<IHH", b[4:12])
    return [b[largo_encabezado + i * largo_registro + 1:largo_encabezado + (i + 1) * largo_registro].decode().strip()
            for i in range(n)]


def a_grados(xy):
    """WGS 84 World Mercator (EPSG:3395, elipsoidal) a longitud y latitud en grados."""
    lon = np.degrees(xy[:, 0] / R_WGS84)
    t = np.exp(-xy[:, 1] / R_WGS84)
    phi = np.pi / 2 - 2 * np.arctan(t)
    for _ in range(10):   # converge a milímetros en 4 o 5 iteraciones
        e_sin = EXCENTRICIDAD_WGS84 * np.sin(phi)
        phi = np.pi / 2 - 2 * np.arctan(t * ((1 - e_sin) / (1 + e_sin)) ** (EXCENTRICIDAD_WGS84 / 2))
    return np.column_stack([lon, np.degrees(phi)])


def grilla_era5l():
    """Centros de celda (latitud, longitud) y máscara de tierra de la grilla descargada."""
    with netCDF4.Dataset(DIR_ERA5L / "acumuladas" / "acumuladas_1981.nc") as ds:
        lat = np.round(ds.variables["latitude"][:].astype(float), 2)
        lon = np.round(ds.variables["longitude"][:].astype(float), 2)
        tierra = ~np.isnan(np.ma.filled(ds.variables["tp"][0].astype("float32"), np.nan))
    return lat, lon, tierra


def area_celda_km2(lat):
    """Área de una celda de 0,1° × 0,1° centrada en `lat`, sobre el elipsoide WGS84."""
    a, e2 = 6378.137, 0.00669437999014
    phi = np.radians(lat)
    s2 = np.sin(phi) ** 2
    m = a * (1 - e2) / (1 - e2 * s2) ** 1.5      # radio de curvatura del meridiano
    n = a / np.sqrt(1 - e2 * s2)                  # radio de curvatura del primer vertical
    d = np.radians(PASO_ERA5L)
    return m * d * n * np.cos(phi) * d


def fracciones(anillos, lon_c, lat_c):
    """Fracción de cada celda (centros lon_c × lat_c) dentro del polígono. Barrido por filas con la regla par-impar:
    un punto está dentro si a su izquierda hay un número impar de cruces con los bordes de todos los anillos."""
    off = (np.arange(SUB_CELDA) + 0.5) / SUB_CELDA * PASO_ERA5L - PASO_ERA5L / 2
    xs, ys = (lon_c[:, None] + off).ravel(), (lat_c[:, None] + off).ravel()
    lados = []
    for a in anillos:
        if not np.allclose(a[0], a[-1]):
            a = np.vstack([a, a[:1]])
        lados.append(np.column_stack([a[:-1], a[1:]]))
    x1, y1, x2, y2 = np.vstack(lados).T
    orden = np.argsort(np.minimum(y1, y2))
    x1, y1, x2, y2 = x1[orden], y1[orden], x2[orden], y2[orden]
    ymin, ymax = np.minimum(y1, y2), np.maximum(y1, y2)
    dentro = np.zeros((len(ys), len(xs)), bool)
    hasta = np.searchsorted(ymin, ys, side="right")   # solo los lados que empiezan por debajo de la fila
    for i, y in enumerate(ys):
        k = hasta[i]
        sel = (ymax[:k] > y) & ((y1[:k] <= y) != (y2[:k] <= y))
        if not sel.any():
            continue
        xa, ya, xb, yb = x1[:k][sel], y1[:k][sel], x2[:k][sel], y2[:k][sel]
        xi = np.sort(xa + (y - ya) * (xb - xa) / (yb - ya))
        dentro[i] = np.searchsorted(xi, xs) % 2 == 1
    return dentro.reshape(len(lat_c), SUB_CELDA, len(lon_c), SUB_CELDA).mean(axis=(1, 3))


def poligonos_grados():
    pol = dict(zip(leer_ids_dbf(DIR_LIMITES / "CAMELS_COL_catchments_boundaries.dbf"),
                   leer_shp(DIR_LIMITES / "CAMELS_COL_catchments_boundaries.shp")))
    faltan = [g for g in IDS_ERA5L if g not in pol]
    assert not faltan, f"Cuencas sin polígono: {faltan}"
    return {g: [a_grados(a) for a in pol[g]] for g in IDS_ERA5L}


def calcular_pesos():
    lat, lon, tierra = grilla_era5l()
    pol = poligonos_grados()
    atr = pd.read_csv(DIR_SALIDA / "atributos.csv", dtype={"gauge_id": str}).set_index("gauge_id")
    partes, resumen_g = [], []
    for g in IDS_ERA5L:
        xy = np.vstack(pol[g])
        assert (xy[:, 0].min() >= lon.min() - PASO_ERA5L / 2 and xy[:, 0].max() <= lon.max() + PASO_ERA5L / 2 and
                xy[:, 1].min() >= lat.min() - PASO_ERA5L / 2 and xy[:, 1].max() <= lat.max() + PASO_ERA5L / 2), \
            f"{g} se sale de la caja descargada"
        ic = np.where((lon + PASO_ERA5L / 2 > xy[:, 0].min()) & (lon - PASO_ERA5L / 2 < xy[:, 0].max()))[0]
        ir = np.where((lat + PASO_ERA5L / 2 > xy[:, 1].min()) & (lat - PASO_ERA5L / 2 < xy[:, 1].max()))[0]
        f = fracciones(pol[g], lon[ic], lat[ir])
        r, q = np.nonzero(f > 0)
        t = pd.DataFrame({"gauge_id": g, "fila": ir[r], "col": ic[q], "lat": lat[ir[r]], "lon": lon[ic[q]],
                          "fraccion": f[r, q]})
        t["area_dentro_km2"] = t["fraccion"] * area_celda_km2(t["lat"].to_numpy())
        t["tierra"] = tierra[t["fila"], t["col"]]
        partes.append(t)
        area_g = t["area_dentro_km2"].sum()
        resumen_g.append({"gauge_id": g, "area_atributo_km2": atr.loc[g, "area"], "area_poligono_km2": area_g,
                          "razon_area": area_g / atr.loc[g, "area"], "celdas": len(t),
                          "celdas_tierra": int(t["tierra"].sum()),
                          "frac_area_en_mar": t.loc[~t["tierra"], "area_dentro_km2"].sum() / area_g,
                          "celda_mayor_frac": t["fraccion"].max()})
    p, s = pd.concat(partes, ignore_index=True), pd.DataFrame(resumen_g)
    assert (s["celdas_tierra"] > 0).all(), "Hay cuencas sin celdas de tierra"
    p["peso"] = np.where(p["tierra"], p["area_dentro_km2"], 0.0)
    p["peso"] = p["peso"] / p.groupby("gauge_id")["peso"].transform("sum")
    p.to_csv(PESOS_ERA5L, index=False, float_format="%.6g")
    s.to_csv(RESUMEN_PESOS_ERA5L, index=False, float_format="%.6g")


if EJECUTAR_PESOS_ERA5L:
    calcular_pesos()

if PESOS_ERA5L.exists():
    pesos_era5l = pd.read_csv(PESOS_ERA5L, dtype={"gauge_id": str})
    resumen_pesos = pd.read_csv(RESUMEN_PESOS_ERA5L, dtype={"gauge_id": str})
    print(f"{len(pesos_era5l):,} pares cuenca-celda; {pesos_era5l.loc[pesos_era5l['peso'] > 0, ['fila', 'col']].drop_duplicates().shape[0]:,} "
          f"celdas distintas con peso; cuencas con parte en celdas de mar: {int((resumen_pesos['frac_area_en_mar'] > 0).sum())}")
    display(resumen_pesos[["razon_area", "celdas", "celda_mayor_frac"]].describe(percentiles=[.05, .5, .95]).round(3).T)
else:
    print("Pesos pendientes: cambia EJECUTAR_PESOS_ERA5L a True")

23,067 pares cuenca-celda; 5,569 celdas distintas con peso; cuencas con parte en celdas de mar: 0


,count,mean,std,min,5%,50%,95%,max
razon_area,248.0,0.990,0.008,0.948,0.978,0.992,1.0,1.008
celdas,248.0,93.012,224.494,3.000,4.000,16.000,499.1,2261.000
celda_mayor_frac,248.0,0.880,0.200,0.095,0.428,1.000,1.0,1.000


### 2.3 Series diarias por cuenca

Dos pasos.
1. **Promedio por cuenca:** cada archivo descargado se multiplica por la matriz de pesos, a la resolución de tiempo original. Queda un `.npz` por archivo en `data/externos/era5land/cuencas/`.
   - Lee de a 366 pasos de tiempo, para cuidar la RAM.
   - Salta los archivos ya procesados, así que sirve a medida que llegan años.
2. **Día local (UTC−5, sin horario de verano):** fecha = piso(hora UTC − 5 h). Por tipo:
   - **acumuladas:** el valor de las 00 UTC del día D + 1 cubre de las 19:00 del día D − 1 a las 19:00 del día D, en hora local;
   - **suelo:** el valor de las 12 UTC, las 07:00 hora local;
   - **instantáneas:** el promedio de las 4 muestras del día local, a las 01, 07, 13 y 19 h.

   Un día solo se acepta con todas sus muestras. La única excepción es el 31/12/2022 de las instantáneas, que tiene 3 porque faltaría el 1/1/2023 a las 00 UTC.

El viento y la presión de vapor se calculan en cada celda y cada hora antes de promediar, porque no son lineales.

Unos 5 minutos en total. Para recalcular, cambia `EJECUTAR_SERIES_ERA5L` a `True`.

In [ ]:
EJECUTAR_SERIES_ERA5L = False
DIR_INTER_ERA5L = DIR_ERA5L / "cuencas"
TABLA_ERA5L = DIR_ERA5L / "era5land_cuencas_diario.parquet"
DESFASE_LOCAL = pd.Timedelta(hours=5)                  # Colombia: UTC−5
INICIO_ERA5L, FIN_ERA5L = CALENDARIO[0], CALENDARIO[-1]
BLOQUE_TIEMPO = 366                                    # pasos de tiempo leídos a la vez
# variable de salida: (tipo de pedido, variable de origen, conversión)
SALIDAS_ERA5L = {
    "era5l_pr_mm":        ("acumuladas", "tp",   lambda x: x * 1000),
    "era5l_srad_wm2":     ("acumuladas", "ssrd", lambda x: x / 86400),
    "era5l_lrad_wm2":     ("acumuladas", "strd", lambda x: x / 86400),
    "era5l_evap_mm":      ("acumuladas", "e",    lambda x: -x * 1000),   # positiva = sale agua
    "era5l_esc_total_mm": ("acumuladas", "ro",   lambda x: x * 1000),
    "era5l_esc_sup_mm":   ("acumuladas", "sro",  lambda x: x * 1000),
    "era5l_esc_sub_mm":   ("acumuladas", "ssro", lambda x: x * 1000),
    "era5l_suelo1":       ("suelo", "swvl1", lambda x: x),
    "era5l_suelo2":       ("suelo", "swvl2", lambda x: x),
    "era5l_suelo3":       ("suelo", "swvl3", lambda x: x),
    "era5l_suelo4":       ("suelo", "swvl4", lambda x: x),
    "era5l_rocio_c":      ("instantaneas", "d2m", lambda x: x - 273.15),
    "era5l_pvapor_kpa":   ("instantaneas", "ea",  lambda x: x),
    "era5l_viento_ms":    ("instantaneas", "ws",  lambda x: x),
    "era5l_presion_kpa":  ("instantaneas", "sp",  lambda x: x / 1000),
}
MUESTRAS_POR_DIA = {"acumuladas": 1, "suelo": 1, "instantaneas": 4}


def matriz_pesos_era5l():
    """Columnas de la grilla aplanada que usa alguna cuenca, y matriz de pesos (celdas × cuencas)."""
    p = pd.read_csv(PESOS_ERA5L, dtype={"gauge_id": str})
    p = p[p["peso"] > 0]
    _, lon, _ = grilla_era5l()
    plano = p["fila"].to_numpy() * len(lon) + p["col"].to_numpy()
    usadas = np.unique(plano)
    w = np.zeros((len(usadas), len(IDS_ERA5L)), "float64")
    w[np.searchsorted(usadas, plano), p["gauge_id"].map({g: i for i, g in enumerate(IDS_ERA5L)}).to_numpy()] = p["peso"]
    return usadas, w


def presion_vapor_kpa(d2m_k):
    td = d2m_k - 273.15
    return 0.6108 * np.exp(17.27 * td / (td + 237.3))   # FAO-56, ec. 14


def promediar_por_cuenca():
    usadas, w = matriz_pesos_era5l()
    DIR_INTER_ERA5L.mkdir(exist_ok=True)
    t_pesos, hechos = PESOS_ERA5L.stat().st_mtime, 0

    def leer(ds, v, i, j):
        return np.ma.filled(ds.variables[v][i:j].astype("float32"), np.nan).reshape(j - i, -1)[:, usadas]

    for tipo in MUESTRAS_POR_DIA:
        for nc in sorted((DIR_ERA5L / tipo).glob(f"{tipo}_*.nc")):
            destino = DIR_INTER_ERA5L / f"{nc.stem}.npz"
            if destino.exists() and destino.stat().st_mtime > max(nc.stat().st_mtime, t_pesos):
                continue
            with netCDF4.Dataset(nc) as ds:
                tiempo = ds.variables["valid_time"][:].astype("int64")   # segundos desde 1970, UTC
                origen = [v for tp, v, _ in SALIDAS_ERA5L.values() if tp == tipo]
                partes = {v: [] for v in origen}
                for i in range(0, len(tiempo), BLOQUE_TIEMPO):
                    j = min(len(tiempo), i + BLOQUE_TIEMPO)
                    for v in origen:
                        if v == "ea":
                            x = presion_vapor_kpa(leer(ds, "d2m", i, j))
                        elif v == "ws":
                            x = np.hypot(leer(ds, "u10", i, j), leer(ds, "v10", i, j))
                        else:
                            x = leer(ds, v, i, j)
                        partes[v].append((x.astype("float64") @ w).astype("float32"))
            tmp = destino.with_name(destino.stem + "_tmp.npz")
            np.savez_compressed(tmp, tiempo=tiempo, **{v: np.vstack(a) for v, a in partes.items()})
            os.replace(tmp, destino)
            hechos += 1
    print(f"Archivos procesados: {hechos}; total en {DIR_INTER_ERA5L.relative_to(ROOT)}: "
          f"{len(list(DIR_INTER_ERA5L.glob('*.npz')))}")


def series_diarias_era5l():
    fechas_d = pd.date_range(INICIO_ERA5L, FIN_ERA5L, freq="D")
    tablas = {}
    for tipo in MUESTRAS_POR_DIA:
        datos = [np.load(a) for a in sorted(DIR_INTER_ERA5L.glob(f"{tipo}_*.npz"))]
        tiempo = np.concatenate([d["tiempo"] for d in datos])
        orden = np.argsort(tiempo, kind="stable")
        orden = orden[np.unique(tiempo[orden], return_index=True)[1]]
        fecha = (pd.to_datetime(tiempo[orden], unit="s") - DESFASE_LOCAL).floor("D")
        n_muestras = pd.Series(1, index=fecha).groupby(level=0).size()
        completo = n_muestras >= MUESTRAS_POR_DIA[tipo]
        if tipo == "instantaneas" and FIN_ERA5L in n_muestras.index and n_muestras[FIN_ERA5L] == 3:
            completo[FIN_ERA5L] = True   # el 31/12/2022 no tiene la muestra de las 19 h
        dias_ok = completo[completo].index
        for salida, (tp, v, conv) in SALIDAS_ERA5L.items():
            if tp == tipo:
                x = np.concatenate([d[v] for d in datos])[orden]
                diaria = pd.DataFrame(x, index=fecha, columns=IDS_ERA5L).groupby(level=0).mean()
                tablas[salida] = conv(diaria.reindex(dias_ok).reindex(fechas_d).to_numpy("float64")).astype("float32")
        for d in datos:
            d.close()
    largo = pd.DataFrame({"gauge_id": pd.Categorical(np.repeat(IDS_ERA5L, len(fechas_d)), categories=IDS_ERA5L),
                          "date": np.tile(fechas_d.to_numpy(), len(IDS_ERA5L))})
    for salida in SALIDAS_ERA5L:
        largo[salida] = tablas.pop(salida).T.ravel()
    tmp = TABLA_ERA5L.with_suffix(".tmp")
    largo.to_parquet(tmp, index=False)
    os.replace(tmp, TABLA_ERA5L)


if EJECUTAR_SERIES_ERA5L:
    promediar_por_cuenca()
    series_diarias_era5l()

if TABLA_ERA5L.exists():
    tabla_era5l = pd.read_parquet(TABLA_ERA5L)
    print(f"{TABLA_ERA5L.relative_to(ROOT)}: {len(tabla_era5l):,} filas = {tabla_era5l['gauge_id'].nunique()} cuencas × "
          f"{tabla_era5l['date'].nunique():,} días ({tabla_era5l['date'].min():%Y-%m-%d} a {tabla_era5l['date'].max():%Y-%m-%d})")
    d = tabla_era5l[list(SALIDAS_ERA5L)].describe(percentiles=[.01, .5, .99]).T[["min", "1%", "50%", "99%", "max"]]
    d["vacíos (%)"] = 100 * tabla_era5l[list(SALIDAS_ERA5L)].isna().mean()
    display(d.round(3))
else:
    print("Series pendientes: cambia EJECUTAR_SERIES_ERA5L a True")

data\externos\era5land\era5land_cuencas_diario.parquet: 3,804,320 filas = 248 cuencas × 15,340 días (1981-01-01 a 2022-12-31)


,min,1%,50%,99%,max,vacíos (%)
era5l_pr_mm,0.000,0.032,6.950,54.070,288.260,0.0
era5l_srad_wm2,11.859,83.369,198.892,287.473,342.580,0.0
era5l_lrad_wm2,237.002,311.825,360.492,421.876,445.141,0.0
era5l_evap_mm,0.191,1.588,3.187,4.700,9.178,0.0
era5l_esc_total_mm,0.000,0.042,4.107,46.337,288.368,0.0
era5l_esc_sup_mm,0.000,0.001,0.612,19.298,201.330,0.0
era5l_esc_sub_mm,0.000,0.021,3.245,31.196,153.110,0.0
era5l_suelo1,0.120,0.212,0.428,0.492,0.520,0.0
era5l_suelo2,0.143,0.226,0.429,0.494,0.520,0.0
era5l_suelo3,0.148,0.247,0.425,0.500,0.520,0.0


### 2.4 Validación

Antes de usar ERA5-Land en el modelo, se comprueba que la tabla esté bien calculada.

1. **Completitud y orden:** una fila por cuenca y día, sin vacíos, en el mismo orden que `camels_col_diario.parquet`.
2. **Recálculo independiente desde las grillas originales**, para 9 cuencas al azar y fechas de borde (1/1/1981, 31/12/1999, 29/2/2000, 31/12/2012 y 31/12/2015):
   - otra forma de medir la fracción (`contains_points` de matplotlib);
   - peso por coseno de la latitud;
   - horas UTC escogidas a mano.

   Tarda unos minutos; se repite con `EJECUTAR_RECALCULO_ERA5L = True`. Si no, se muestra el resultado guardado.
3. **Controles físicos:**
   - balance de agua del propio ERA5-Land;
   - las tres escorrentías;
   - presión frente a la elevación de la cuenca;
   - radiación frente a la del tope de la atmósfera;
   - rocío frente a las temperaturas de CAMELS-COL.
4. **Otras fuentes:**
   - alineación de días con la lluvia de CAMELS-COL (CHIRPS);
   - la lluvia de ERA5 del piloto (Open-Meteo, 30 cuencas, 2001–2010);
   - la escorrentía frente al caudal observado. Este control es solo para evaluar, nunca como entrada.

**Resultado** (18 y 19/11/2026; bitácora, secciones 17 y 18):
- **El recálculo coincide:** la diferencia máxima es de 7 × 10⁻⁵ desviaciones estándar.
- **Los controles físicos cierran:**
  - el balance da −0,1 %;
  - la presión frente a la elevación da r = 0,994;
  - la radiación queda entre 0,12 y 0,74 de la del tope de la atmósfera.
- **Los días están bien alineados:** la correlación con CHIRPS es máxima sin desfase.
- **El problema conocido es el exceso de lluvia del reanálisis:**
  - ERA5-Land llueve 1,5 veces lo de CHIRPS en la mediana de cuencas, con Spearman entre cuencas de 0,30;
  - su escorrentía no sirve como simulación directa: NSE mediano −3,2 frente al caudal.

  Si aporta, es por su dinámica.

In [ ]:
from matplotlib.path import Path as Trazo

EJECUTAR_RECALCULO_ERA5L = False
ARCHIVO_RECALCULO = DIR_ERA5L / "validacion_recalculo.csv"
tabla_era5l["gauge_id"] = tabla_era5l["gauge_id"].astype(str)
VARS_ERA5L = list(SALIDAS_ERA5L)
atr_era5l = pd.read_csv(DIR_SALIDA / "atributos.csv", dtype={"gauge_id": str}).set_index("gauge_id")
cam = pd.read_parquet(DIR_SALIDA / "camels_col_diario.parquet",
                      columns=["gauge_id", "date", "periodo", "streamflow", "pr", "poten_evapo", "t_min", "t_max"])
cam["gauge_id"] = cam["gauge_id"].astype(str)

# 1. Completitud y orden
print("1. Completitud")
print(f"   {len(tabla_era5l):,} filas = {tabla_era5l['gauge_id'].nunique()} cuencas × {tabla_era5l['date'].nunique():,} días: "
      f"{len(tabla_era5l) == tabla_era5l['gauge_id'].nunique() * tabla_era5l['date'].nunique()}; duplicados: "
      f"{int(tabla_era5l.duplicated(['gauge_id', 'date']).sum())}; vacíos: {int(tabla_era5l[VARS_ERA5L].isna().sum().sum())}; "
      f"mismo orden que camels_col_diario.parquet: "
      f"{bool((cam['gauge_id'].to_numpy() == tabla_era5l['gauge_id'].to_numpy()).all() and (cam['date'].to_numpy() == tabla_era5l['date'].to_numpy()).all())}")


# 2. Recálculo independiente desde las grillas
def recalcular_era5l():
    rng = np.random.default_rng(20261118)
    fin_val = pd.Timestamp("2015-12-31")
    pol = dict(zip(leer_ids_dbf(DIR_LIMITES / "CAMELS_COL_catchments_boundaries.dbf"),
                   leer_shp(DIR_LIMITES / "CAMELS_COL_catchments_boundaries.shp")))
    lat, lon, _ = grilla_era5l()

    def pesos_independientes(g):
        anillos = [a_grados(a) for a in pol[g]]
        trazos, xy = [Trazo(a) for a in anillos], np.vstack(anillos)
        off = np.linspace(-0.05 + 0.0025, 0.05 - 0.0025, 20)
        w = np.zeros((len(lat), len(lon)))
        for i in np.where((lat + 0.05 > xy[:, 1].min()) & (lat - 0.05 < xy[:, 1].max()))[0]:
            for j in np.where((lon + 0.05 > xy[:, 0].min()) & (lon - 0.05 < xy[:, 0].max()))[0]:
                px, py = np.meshgrid(lon[j] + off, lat[i] + off)
                pts = np.column_stack([px.ravel(), py.ravel()])
                d = np.zeros(len(pts), bool)
                for tr in trazos:
                    d ^= tr.contains_points(pts)
                w[i, j] = d.mean() * np.cos(np.radians(lat[i]))
        return w

    def campo(tipo, var, momento):
        nc = DIR_ERA5L / tipo / f"{tipo}_{momento.year}.nc"
        if tipo == "acumuladas" and momento == pd.Timestamp("2023-01-01"):
            nc = DIR_ERA5L / "acumuladas" / "acumuladas_2023_01_01.nc"
        with netCDF4.Dataset(nc) as ds:
            k = np.where(pd.to_datetime(ds.variables["valid_time"][:].astype("int64"), unit="s") == momento)[0]
            assert len(k) == 1, (nc.name, momento)
            return np.ma.filled(ds.variables[var][k[0]].astype("float64"), np.nan)

    def promedio(w, x):
        ok = ~np.isnan(x) & (w > 0)
        return (w[ok] * x[ok]).sum() / w[ok].sum()

    def valores(d, w):
        sig, out = d + pd.Timedelta(days=1), {}
        acum = {"era5l_pr_mm": ("tp", 1000), "era5l_srad_wm2": ("ssrd", 1 / 86400), "era5l_lrad_wm2": ("strd", 1 / 86400),
                "era5l_evap_mm": ("e", -1000), "era5l_esc_total_mm": ("ro", 1000), "era5l_esc_sup_mm": ("sro", 1000),
                "era5l_esc_sub_mm": ("ssro", 1000)}
        for k, (v, f) in acum.items():
            out[k] = promedio(w, campo("acumuladas", v, sig)) * f       # 00 UTC del día siguiente = total del día
        for n in range(1, 5):
            out[f"era5l_suelo{n}"] = promedio(w, campo("suelo", f"swvl{n}", d + pd.Timedelta(hours=12)))
        horas = [d + pd.Timedelta(hours=h) for h in (6, 12, 18)] + [sig]   # 01, 07, 13 y 19 h local
        td = [campo("instantaneas", "d2m", h) - 273.15 for h in horas]
        out["era5l_rocio_c"] = np.mean([promedio(w, x) for x in td])
        out["era5l_pvapor_kpa"] = np.mean([promedio(w, 0.6108 * np.exp(17.27 * x / (x + 237.3))) for x in td])
        out["era5l_viento_ms"] = np.mean([promedio(w, np.hypot(campo("instantaneas", "u10", h),
                                                                campo("instantaneas", "v10", h))) for h in horas])
        out["era5l_presion_kpa"] = np.mean([promedio(w, campo("instantaneas", "sp", h)) for h in horas]) / 1000
        return out

    ids_era5l = sorted(tabla_era5l["gauge_id"].unique())
    pequenas = resumen_pesos.set_index("gauge_id").query("celdas <= 150").index.intersection(ids_era5l)
    cuencas_r = list(rng.choice(pequenas, 8, replace=False)) + ["52017080"]
    dias = list(pd.to_datetime(rng.choice(pd.date_range("1981-01-01", fin_val).to_numpy(), 6, replace=False)))
    dias += [pd.Timestamp(x) for x in ("1981-01-01", "1999-12-31", "2000-02-29", "2012-12-31", "2015-12-31")]
    filas_r = []
    for g in cuencas_r:
        w = pesos_independientes(g)
        for k, d in enumerate(dias):
            if k % 2 and d.year not in (1981, 2000, 2015):   # no todas las combinaciones, para no tardar
                continue
            ind = valores(d, w)
            tab = tabla_era5l[(tabla_era5l["gauge_id"] == g) & (tabla_era5l["date"] == d)].iloc[0]
            filas_r += [{"gauge_id": g, "date": d, "variable": v, "tabla": float(tab[v]), "independiente": ind[v]}
                        for v in VARS_ERA5L]
    comp = pd.DataFrame(filas_r)
    comp["dif_abs"] = (comp["tabla"] - comp["independiente"]).abs()
    escala = tabla_era5l.loc[tabla_era5l["date"] <= fin_val, VARS_ERA5L].std()
    comp["dif_rel_escala"] = comp["dif_abs"] / comp["variable"].map(escala)
    comp.to_csv(ARCHIVO_RECALCULO, index=False)


if EJECUTAR_RECALCULO_ERA5L:
    recalcular_era5l()
comp = pd.read_csv(ARCHIVO_RECALCULO, dtype={"gauge_id": str})
print(f"\n2. Recálculo independiente: {comp[['gauge_id', 'date']].drop_duplicates().shape[0]} pares cuenca-fecha de "
      f"{comp['gauge_id'].nunique()} cuencas, {len(comp)} valores; diferencia máxima "
      f"{comp['dif_rel_escala'].max():.1e} desviaciones estándar ({comp.loc[comp['dif_rel_escala'].idxmax(), 'variable']})")

# 3. Controles físicos
m = tabla_era5l.groupby("gauge_id")[VARS_ERA5L].mean()
bal = (m["era5l_pr_mm"] - m["era5l_evap_mm"] - m["era5l_esc_total_mm"]) / m["era5l_pr_mm"]
z = atr_era5l.loc[m.index, "mean_ele"]
p_fao = 101.3 * ((293 - 0.0065 * z) / 293) ** 5.26             # FAO-56, ec. 7
print("\n3. Controles físicos (1981–2022)")
print(f"   balance (lluvia − evaporación − escorrentía) / lluvia, por cuenca (p5, p50, p95): {bal.quantile([.05, .5, .95]).round(3).tolist()}")
print(f"   escorrentía superficial + subsuperficial − total: máximo {(m['era5l_esc_sup_mm'] + m['era5l_esc_sub_mm'] - m['era5l_esc_total_mm']).abs().max():.1e} mm/día")
print(f"   presión media frente a la de la elevación media (FAO-56): r = {np.corrcoef(m['era5l_presion_kpa'], p_fao)[0, 1]:.3f}, "
      f"diferencia mediana {np.median(m['era5l_presion_kpa'] - p_fao):+.2f} kPa")
grados = pd.DataFrame(a_grados(atr_era5l[["gauge_lon", "gauge_lat"]].to_numpy(float)), index=atr_era5l.index,
                      columns=["lon", "lat"])                   # las estaciones vienen en metros (Mercator)
phi = np.radians(grados.loc[tabla_era5l["gauge_id"], "lat"].to_numpy())
J = tabla_era5l["date"].dt.dayofyear.to_numpy()
dr, dec = 1 + 0.033 * np.cos(2 * np.pi * J / 365), 0.409 * np.sin(2 * np.pi * J / 365 - 1.39)
ws = np.arccos(-np.tan(phi) * np.tan(dec))
ra = 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))  # FAO-56, ec. 21
k_rad = tabla_era5l["era5l_srad_wm2"].to_numpy() / (ra * 1e6 / 86400)
print(f"   radiación solar / radiación del tope de la atmósfera, por día (p0,1, p50, p99,9): "
      f"{np.percentile(k_rad, 0.1):.2f}, {np.median(k_rad):.2f}, {np.percentile(k_rad, 99.9):.2f}")
ok = cam["t_max"].notna().to_numpy()
print(f"   días con rocío de ERA5-Land mayor que la t_max de CAMELS-COL: "
      f"{np.mean(tabla_era5l['era5l_rocio_c'].to_numpy()[ok] > cam['t_max'].to_numpy()[ok]):.2%}")

# 4. Otras fuentes
cr = cam[["gauge_id", "date", "pr"]].assign(e=tabla_era5l["era5l_pr_mm"].to_numpy())
desfases = {}
for lag in range(-2, 3):
    rs = []
    for g, t in cr.groupby("gauge_id"):
        a, b = t["pr"].to_numpy(), t["e"].shift(lag).to_numpy()
        okk = ~np.isnan(a) & ~np.isnan(b)
        if okk.sum() > 365:
            rs.append(np.corrcoef(a[okk], b[okk])[0, 1])
    desfases[lag] = round(float(np.median(rs)), 3)
anual = cr.groupby("gauge_id")[["pr", "e"]].mean() * 365.25
print("\n4. Otras fuentes")
print(f"   correlación diaria mediana con la lluvia de CAMELS-COL según desfase (días; + = ERA5-Land corrido hacia adelante): {desfases}")
print(f"   lluvia anual, mediana de cuencas: CAMELS-COL {anual['pr'].median():.0f} mm, ERA5-Land {anual['e'].median():.0f} mm; "
      f"razón (p5, p50, p95): {(anual['e'] / anual['pr']).quantile([.05, .5, .95]).round(2).tolist()}; "
      f"Spearman entre cuencas: {anual['pr'].corr(anual['e'], method='spearman'):.2f}")
om = [pd.read_csv(f, parse_dates=["date"]).assign(gauge_id=f.stem)
      for f in sorted((ROOT / "data" / "externos" / "era5_openmeteo").glob("*.csv")) if f.stem.isdigit()]
if om:
    om = pd.concat(om).merge(tabla_era5l[["gauge_id", "date", "era5l_pr_mm"]], on=["gauge_id", "date"])
    r_om = [np.corrcoef(x["pr_era5"], x["era5l_pr_mm"])[0, 1] for _, x in om.dropna(subset=["pr_era5"]).groupby("gauge_id")]
    print(f"   frente a ERA5 de Open-Meteo ({om['gauge_id'].nunique()} cuencas, 2001–2010): correlación diaria mediana "
          f"{np.median(r_om):.2f}")
q = cam.assign(esc=tabla_era5l["era5l_esc_total_mm"].to_numpy())
q = q[(q["periodo"] == "train") & q["streamflow"].notna()]
nse_q = q.groupby("gauge_id").apply(lambda x: 1 - ((x["esc"] - x["streamflow"]) ** 2).sum()
                                    / ((x["streamflow"] - x["streamflow"].mean()) ** 2).sum())
print(f"   escorrentía de ERA5-Land frente al caudal observado (1981–2010, solo para evaluar): NSE mediano {nse_q.median():.2f}, "
      f"volumen {(q.groupby('gauge_id')['esc'].mean() / q.groupby('gauge_id')['streamflow'].mean()).median():.2f} veces el observado")
del cam, cr, q

1. Completitud


   3,804,320 filas = 248 cuencas × 15,340 días: True; duplicados: 0; vacíos: 0; mismo orden que camels_col_diario.parquet: True

2. Recálculo independiente: 54 pares cuenca-fecha de 9 cuencas, 810 valores; diferencia máxima 7.1e-05 desviaciones estándar (era5l_rocio_c)



3. Controles físicos (1981–2022)
   balance (lluvia − evaporación − escorrentía) / lluvia, por cuenca (p5, p50, p95): [-0.005, -0.0, 0.001]
   escorrentía superficial + subsuperficial − total: máximo 3.8e-06 mm/día
   presión media frente a la de la elevación media (FAO-56): r = 0.994, diferencia mediana +0.99 kPa


   radiación solar / radiación del tope de la atmósfera, por día (p0,1, p50, p99,9): 0.12, 0.48, 0.74
   días con rocío de ERA5-Land mayor que la t_max de CAMELS-COL: 0.04%



4. Otras fuentes
   correlación diaria mediana con la lluvia de CAMELS-COL según desfase (días; + = ERA5-Land corrido hacia adelante): {-2: 0.18, -1: 0.229, 0: 0.336, 1: 0.189, 2: 0.133}


   lluvia anual, mediana de cuencas: CAMELS-COL 1997 mm, ERA5-Land 2964 mm; razón (p5, p50, p95): [0.71, 1.56, 3.83]; Spearman entre cuencas: 0.29


   frente a ERA5 de Open-Meteo (30 cuencas, 2001–2010): correlación diaria mediana 0.93


   escorrentía de ERA5-Land frente al caudal observado (1981–2010, solo para evaluar): NSE mediano -3.19, volumen 1.60 veces el observado


### 2.5 Base para el LSTM con ERA5-Land

Copia de la base de 1.12 con las 15 variables de ERA5-Land agregadas a la serie de cada cuenca, en `data/processed_era5land/`. No toca `data/processed`.
- Los atributos son los mismos: se copia `atributos_modelo.csv`. El modelado quita `eroded_misce_perc` desde su configuración (8.6).
- El LSTM la usa en las secciones 8.6 y 8.7 del modelado.
- Unos 2 minutos. Para reconstruirla, cambia `EJECUTAR_BASE_ERA5L` a `True`. No la reconstruyas mientras un entrenamiento la está leyendo.

In [ ]:
import shutil

import xarray as xr

EJECUTAR_BASE_ERA5L = False
DIR_NH_ERA5L = ROOT / "data" / "processed_era5land"


def construir_base_era5l():
    (DIR_NH_ERA5L / "time_series").mkdir(parents=True, exist_ok=True)
    (DIR_NH_ERA5L / "attributes").mkdir(exist_ok=True)
    shutil.copy2(DIR_SALIDA / "attributes" / "atributos_modelo.csv", DIR_NH_ERA5L / "attributes" / "atributos_modelo.csv")
    for g, e in tabla_era5l.groupby("gauge_id", sort=True):
        with xr.open_dataset(DIR_SALIDA / "time_series" / f"{g}.nc") as ds:
            ds = ds.load()
        e = e.set_index("date")[VARS_ERA5L]
        for v in VARS_ERA5L:
            ds[v] = ("date", e[v].reindex(pd.DatetimeIndex(ds["date"].values)).to_numpy("float32"))
        ds.to_netcdf(DIR_NH_ERA5L / "time_series" / f"{g}.nc")


if EJECUTAR_BASE_ERA5L:
    construir_base_era5l()

archivos_era5l = sorted((DIR_NH_ERA5L / "time_series").glob("*.nc"))
if archivos_era5l:
    vacios_base = {}
    for a in archivos_era5l:
        with xr.open_dataset(a) as ds:
            vacios_base[a.stem] = int(sum(ds[v].isnull().sum() for v in VARS_ERA5L))
            variables_base = list(ds.data_vars)
    iguales = (pd.read_csv(DIR_SALIDA / "attributes" / "atributos_modelo.csv").equals(
        pd.read_csv(DIR_NH_ERA5L / "attributes" / "atributos_modelo.csv")))
    print(f"{len(archivos_era5l)} series en {DIR_NH_ERA5L.relative_to(ROOT)}; variables: {variables_base}")
    print(f"Días vacíos de ERA5-Land en todas las series: {sum(vacios_base.values())}; "
          f"atributos iguales a los de data/processed: {iguales}")
else:
    print("Base con ERA5-Land pendiente: cambia EJECUTAR_BASE_ERA5L a True")

248 series en data\processed_era5land; variables: ['pr', 'poten_evapo', 't_min', 't_max', 'streamflow', 'era5l_pr_mm', 'era5l_srad_wm2', 'era5l_lrad_wm2', 'era5l_evap_mm', 'era5l_esc_total_mm', 'era5l_esc_sup_mm', 'era5l_esc_sub_mm', 'era5l_suelo1', 'era5l_suelo2', 'era5l_suelo3', 'era5l_suelo4', 'era5l_rocio_c', 'era5l_pvapor_kpa', 'era5l_viento_ms', 'era5l_presion_kpa']
Días vacíos de ERA5-Land en todas las series: 0; atributos iguales a los de data/processed: True
